In [1]:
#### ext   



# scaler = torch.cuda.amp.GradScaler()

# for images, labels in train_loader:
#     optimizer.zero_grad()
#     with torch.cuda.amp.autocast():
#         outputs = model(images)
#         loss = criterion(outputs, labels)
#     scaler.scale(loss).backward()
#     scaler.step(optimizer)
#     scaler.update()

In [2]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [3]:
import pandas as pd
import os 
import numpy as np 
import cv2
from sklearn.model_selection import train_test_split 
from torch.utils.data import DataLoader
import torch 
from torchvision import transforms
from model.resnet50_lstm import ShuffleNetLSTM
from data_ingestion.data_import import DeepfakeDataset


In [4]:
### NO NEED OF THIS 


# root = r"C:\Users\rohit\OneDrive\Desktop\DeepFakeDetectionSystem\faceframes"
# for label in ["real","fake"]:
#     input = os.path.join(root,label)
#     for video in os.listdir(input):
#         video_path = os.path.join(input,video)
#         for frames in os.listdir(video_path):
#             frame_path = os.path.join(video_path,frames)

#             img = cv2.imread(frame_path) 
#             if img is None:
#                 continue
            
#             img = cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
#             img = img.astype("float32") /255.0



In [5]:
data = r"C:\Users\rohit\OneDrive\Desktop\DeepFakeDetectionSystem\data.csv"
df = pd.read_csv(data)

In [6]:
df.head()

,orignal_video_path,fake_video_path,original_label,fake_label,pair_id
0,faceframes\real\video1.mp4,faceframes\fake\fakevideo1.mp4,1,0,video1
1,faceframes\real\video10.mp4,faceframes\fake\fakevideo10.mp4,1,0,video10
2,faceframes\real\video100.mp4,faceframes\fake\fakevideo100.mp4,1,0,video100
3,faceframes\real\video1000.mp4,faceframes\fake\fakevideo1000.mp4,1,0,video1000
4,faceframes\real\video101.mp4,faceframes\fake\fakevideo101.mp4,1,0,video101


In [7]:
## Data Transformation
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean= [0.5,0.5,0.5],std=[0.5,0.5,0.5])
])

In [8]:
## spliting the data into train ,test and validation set

train_df,test_df = train_test_split(
    df,
    test_size=0.3,
    random_state=42
)

In [9]:
train_dataset = DeepfakeDataset(
    df=train_df, 
    seq_len=30, 
    transform=transform
)

In [10]:
test_dataset = DeepfakeDataset(
    df = test_df,
    seq_len = 30,
    transform=transform
)

In [11]:
## DataLoader 

train_loader = DataLoader(
    train_dataset,
    batch_size=16,      
    shuffle=True,      
    num_workers=8,     
    pin_memory=True    
)

In [12]:
test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=8,
    pin_memory=True
)

In [13]:
## creating the validation set 

val_df,test_df = train_test_split(
    test_df,
    test_size=0.5,
    random_state=42
)

In [14]:
print(train_df.head(2))
print(test_df.head(2))
print(val_df.head(2))

               orignal_video_path                   fake_video_path  \
541  faceframes\real\video586.mp4  faceframes\fake\fakevideo586.mp4   
440  faceframes\real\video495.mp4  faceframes\fake\fakevideo495.mp4   

     original_label  fake_label   pair_id  
541               1           0  video586  
440               1           0  video495  
               orignal_video_path                   fake_video_path  \
557   faceframes\real\video60.mp4   faceframes\fake\fakevideo60.mp4   
798  faceframes\real\video817.mp4  faceframes\fake\fakevideo817.mp4   

     original_label  fake_label   pair_id  
557               1           0   video60  
798               1           0  video817  
               orignal_video_path                   fake_video_path  \
381  faceframes\real\video441.mp4  faceframes\fake\fakevideo441.mp4   
959  faceframes\real\video962.mp4  faceframes\fake\fakevideo962.mp4   

     original_label  fake_label   pair_id  
381               1           0  video441  
959   

In [15]:
print(train_df.shape,test_df.shape,val_df.shape)

(700, 5) (150, 5) (150, 5)


In [16]:
df.head()

,orignal_video_path,fake_video_path,original_label,fake_label,pair_id
0,faceframes\real\video1.mp4,faceframes\fake\fakevideo1.mp4,1,0,video1
1,faceframes\real\video10.mp4,faceframes\fake\fakevideo10.mp4,1,0,video10
2,faceframes\real\video100.mp4,faceframes\fake\fakevideo100.mp4,1,0,video100
3,faceframes\real\video1000.mp4,faceframes\fake\fakevideo1000.mp4,1,0,video1000
4,faceframes\real\video101.mp4,faceframes\fake\fakevideo101.mp4,1,0,video101


In [17]:
import torch
import torch.nn as nn 
from torchvision import models

In [18]:
shufflenet = models.shufflenet_v2_x1_0(weights="DEFAULT")
shufflenet.fc = nn.Identity()

In [19]:
# resnet = models.resnet50(pretrained = True)
# resnet.fc = nn.Identity()

In [20]:
for params in shufflenet.parameters():
    params.requires_grad = False

In [21]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ShuffleNetLSTM(shufflenet).to(device)


In [22]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr =0.0001)

In [23]:
from tqdm import tqdm

In [25]:
## training

model.train()
for epoch in range(30):
    loop = tqdm(train_df,desc=f"Epoch [{epoch+1}/30]")
    for images,labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs,labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        print(f"The Epoch: {epoch+1} with loss: {loss.item():.4f}")

Epoch [1/30]:   0%|          | 0/700 [01:22<?, ?it/s]


The Epoch: 1 with loss: 0.6818
The Epoch: 1 with loss: 0.6896
The Epoch: 1 with loss: 0.6809
The Epoch: 1 with loss: 0.6837
The Epoch: 1 with loss: 0.7105
The Epoch: 1 with loss: 0.6880
The Epoch: 1 with loss: 0.6936
The Epoch: 1 with loss: 0.7137
The Epoch: 1 with loss: 0.6937
The Epoch: 1 with loss: 0.6934
The Epoch: 1 with loss: 0.6875
The Epoch: 1 with loss: 0.6979
The Epoch: 1 with loss: 0.7035
The Epoch: 1 with loss: 0.6932
The Epoch: 1 with loss: 0.6957
The Epoch: 1 with loss: 0.7005
The Epoch: 1 with loss: 0.6884
The Epoch: 1 with loss: 0.6924
The Epoch: 1 with loss: 0.7028
The Epoch: 1 with loss: 0.6888
The Epoch: 1 with loss: 0.6936
The Epoch: 1 with loss: 0.6944
The Epoch: 1 with loss: 0.6922
The Epoch: 1 with loss: 0.6916
The Epoch: 1 with loss: 0.6923
The Epoch: 1 with loss: 0.6920
The Epoch: 1 with loss: 0.6926
The Epoch: 1 with loss: 0.6892
The Epoch: 1 with loss: 0.6900
The Epoch: 1 with loss: 0.6911
The Epoch: 1 with loss: 0.6939
The Epoch: 1 with loss: 0.6919
The Epoc

Epoch [1/30]:   0%|          | 0/700 [03:52<?, ?it/s]


The Epoch: 2 with loss: 0.6821
The Epoch: 2 with loss: 0.6836
The Epoch: 2 with loss: 0.6806
The Epoch: 2 with loss: 0.6848
The Epoch: 2 with loss: 0.6840
The Epoch: 2 with loss: 0.6830
The Epoch: 2 with loss: 0.6933
The Epoch: 2 with loss: 0.6833
The Epoch: 2 with loss: 0.6685
The Epoch: 2 with loss: 0.6909
The Epoch: 2 with loss: 0.6615
The Epoch: 2 with loss: 0.6789
The Epoch: 2 with loss: 0.7065
The Epoch: 2 with loss: 0.6625
The Epoch: 2 with loss: 0.6804
The Epoch: 2 with loss: 0.6653
The Epoch: 2 with loss: 0.6777
The Epoch: 2 with loss: 0.6603
The Epoch: 2 with loss: 0.6389
The Epoch: 2 with loss: 0.6664
The Epoch: 2 with loss: 0.6139
The Epoch: 2 with loss: 0.5150
The Epoch: 2 with loss: 0.6430
The Epoch: 2 with loss: 0.6660
The Epoch: 2 with loss: 1.0354
The Epoch: 2 with loss: 0.4750
The Epoch: 2 with loss: 0.5877
The Epoch: 2 with loss: 0.4899
The Epoch: 2 with loss: 0.5702
The Epoch: 2 with loss: 0.4884
The Epoch: 2 with loss: 0.7258
The Epoch: 2 with loss: 0.5190
The Epoc

Epoch [2/30]:   0%|          | 0/700 [04:09<?, ?it/s]


The Epoch: 3 with loss: 0.6079
The Epoch: 3 with loss: 0.3593
The Epoch: 3 with loss: 0.4202
The Epoch: 3 with loss: 0.5061
The Epoch: 3 with loss: 0.3965
The Epoch: 3 with loss: 0.6875
The Epoch: 3 with loss: 0.4580
The Epoch: 3 with loss: 0.4232
The Epoch: 3 with loss: 0.3271
The Epoch: 3 with loss: 0.3131
The Epoch: 3 with loss: 0.5195
The Epoch: 3 with loss: 0.5884
The Epoch: 3 with loss: 1.2298
The Epoch: 3 with loss: 0.5566
The Epoch: 3 with loss: 0.6327
The Epoch: 3 with loss: 0.6556
The Epoch: 3 with loss: 0.3674
The Epoch: 3 with loss: 0.5383
The Epoch: 3 with loss: 0.5322
The Epoch: 3 with loss: 0.5793
The Epoch: 3 with loss: 0.6189
The Epoch: 3 with loss: 0.4070
The Epoch: 3 with loss: 0.6681
The Epoch: 3 with loss: 0.5440
The Epoch: 3 with loss: 0.5786
The Epoch: 3 with loss: 0.5739
The Epoch: 3 with loss: 0.5384
The Epoch: 3 with loss: 0.5055
The Epoch: 3 with loss: 0.4597
The Epoch: 3 with loss: 0.5686
The Epoch: 3 with loss: 0.5655
The Epoch: 3 with loss: 0.4715
The Epoc

Epoch [3/30]:   0%|          | 0/700 [05:08<?, ?it/s]


The Epoch: 4 with loss: 0.2724
The Epoch: 4 with loss: 0.7570
The Epoch: 4 with loss: 0.6662
The Epoch: 4 with loss: 0.6136
The Epoch: 4 with loss: 0.5473
The Epoch: 4 with loss: 0.3845
The Epoch: 4 with loss: 0.5015
The Epoch: 4 with loss: 0.4992


RuntimeError: Caught RuntimeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "c:\Users\rohit\miniconda3\envs\venv\Lib\site-packages\torch\utils\data\_utils\worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\rohit\miniconda3\envs\venv\Lib\site-packages\torch\utils\data\_utils\fetch.py", line 57, in fetch
    return self.collate_fn(data)
           ^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\rohit\miniconda3\envs\venv\Lib\site-packages\torch\utils\data\_utils\collate.py", line 401, in default_collate
    return collate(batch, collate_fn_map=default_collate_fn_map)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\rohit\miniconda3\envs\venv\Lib\site-packages\torch\utils\data\_utils\collate.py", line 214, in collate
    return [
           ^
  File "c:\Users\rohit\miniconda3\envs\venv\Lib\site-packages\torch\utils\data\_u

In [ ]:
import pickle

In [ ]:
with open('model_t.pkl','wb') as file:
    pickle.dump(model,file)